In [ ]:
# Ridge refit + Tier-1 strict check on the pinned canonical image (CPU only).
# Clones the public repo at a fixed commit, refits data/arousal_index_ridge.json
# with scripts/fit_arousal_ridge.py, executes notebooks/repro.ipynb, and runs
# Step 1's strict comparison (rerun/step1/attempt2/compare2.py).
import subprocess, os, sys, json, glob, shutil
COMMIT = "89be8a8"
REPO = "/kaggle/working/energy-audit"
W = "/kaggle/working"
def sh(cmd, cwd=None, env=None):
    print("$", cmd); r = subprocess.run(cmd, shell=True, cwd=cwd, env=env, capture_output=True, text=True)
    print(r.stdout[-6000:]); print(r.stderr[-3000:]); r.check_returncode(); return r
sh(f"git clone -q --branch rerun/step1-tier1 https://github.com/ConnerFarrell/energy-audit {REPO}")
sh(f"git checkout -q {COMMIT} && git log -1 --format='%H %s'", cwd=REPO)

In [ ]:
import importlib, platform
env = {"python": sys.version.split()[0], "platform": platform.platform()}
for m in ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "nbconvert", "nbclient", "ipykernel"]:
    try: env[m] = importlib.import_module(m).__version__
    except Exception as e: env[m] = f"unavailable ({type(e).__name__})"
json.dump(env, open(f"{W}/env.json", "w"), indent=2); print(json.dumps(env, indent=2))

In [ ]:
ann = glob.glob("/kaggle/input/**/DEAM_Annotations", recursive=True)
feat = glob.glob("/kaggle/input/**/deam_librosa_features.csv", recursive=True)
assert ann and feat, (ann, feat)
deam_dir = os.path.dirname(ann[0])
sh(f"{sys.executable} scripts/fit_arousal_ridge.py --deam-dir '{deam_dir}' --features '{feat[0]}' --out data/arousal_index_ridge.json", cwd=REPO)
shutil.copy(f"{REPO}/data/arousal_index_ridge.json", f"{W}/arousal_index_ridge.json")
print(open(f"{W}/arousal_index_ridge.json").read())

In [ ]:
e = dict(os.environ, REPRO_OUT_DIR=f"{W}/repro_out")
sh(f"{sys.executable} -m jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --output-dir {W}/repro_exec notebooks/repro.ipynb", cwd=REPO, env=e)
r = sh(f"{sys.executable} rerun/step1/attempt2/compare2.py {W}/repro_out/tables {W}/cell_comparison.csv", cwd=REPO)
open(f"{W}/compare_summary.txt", "w").write(r.stdout)
for t in glob.glob(f"{W}/repro_out/tables/*.csv"): shutil.copy(t, W + "/table__" + os.path.basename(t))
shutil.rmtree(REPO)   # keep only results in the output